In [8]:
import folium
from folium import plugins

# 1. Inisialisasi peta berpusat di Universitas Bengkulu
peta_interaktif = folium.Map(
    location=[-3.7576, 102.2701],
    zoom_start=16,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Imagery'
)

# Menambahkan peta jalan (hybrid) sebagai overlay nama-nama jalan
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Transportation/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Transportation',
    name='Jalan & Label',
    overlay=True,
    control=False
).add_to(peta_interaktif)


# 2. DATA ASET & KERUSAKAN (Simulasi di Area Kampus UNIB)
daftar_aset = [
    {
        "id": "UNIB-JLN-08",
        "kategori": "Jalan Berlubang (Akses Utama)",
        "lokasi": "Dekat Bundaran Utama UNIB",
        "status": "BAHAYA (Menunggu Inspeksi)",
        "status_color": "#d9534f", # Merah
        "koordinat": [-3.7576, 102.2701],
        "icon": "exclamation-sign",
        "marker_color": "red"
    },
    {
        "id": "UNIB-LMP-12",
        "kategori": "Lampu Jalan Padam",
        "lokasi": "Area Parkir Gedung Rektorat",
        "status": "PERLU PENANGANAN (Segera)",
        "status_color": "#f0ad4e", # Kuning / Orange
        "koordinat": [-3.7562, 102.2718],
        "icon": "info-sign",
        "marker_color": "orange"
    },
    {
        "id": "UNIB-PHN-03",
        "kategori": "Pohon Tumbang Sebagian",
        "lokasi": "Jalur Lingkar Dekat Fakultas Teknik",
        "status": "SELESAI (Penanganan Lapangan)",
        "status_color": "#5cb85c", # Hijau
        "koordinat": [-3.7592, 102.2685],
        "icon": "leaf",
        "marker_color": "green"
    }
]


# 3. PENAMBAHAN MARKER INTERAKTIF (Peluang Lulusan IT - Rich Interactive)
for aset in daftar_aset:
    html_popup = f"""
    <div style="font-family: 'Arial', sans-serif; min-width: 250px; border: 1px solid #003366; background-color: #FFFFFF; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">

        <!-- Header Popup -->
        <div style="background-color: #003366; color: #FFFFFF; padding: 12px; text-align: center; border-bottom: 4px solid #FFD700;">
            <h4 style="margin: 0; font-size: 13px; font-weight: bold; letter-spacing: 0.5px;">⚠️ DETAIL LAPORAN ASET</h4>
        </div>

        <!-- Body Informasi -->
        <div style="padding: 12px; text-align: left; color: #333333; line-height: 1.5;">
            <p style="margin: 3px 0; font-size: 12px;"><b>ID Laporan:</b> {aset['id']}</p>
            <p style="margin: 3px 0; font-size: 12px;"><b>Masalah:</b> {aset['kategori']}</p>
            <p style="margin: 3px 0; font-size: 12px;"><b>Lokasi:</b> {aset['lokasi']}</p>
            <p style="margin: 3px 0; font-size: 12px;"><b>Status:</b>
                <span style="background-color: {aset['status_color']}; color: white; padding: 2px 6px; border-radius: 4px; font-size: 10px; font-weight: bold;">
                    {aset['status']}
                </span>
            </p>
        </div>

        <!-- Tombol Aksi -->
        <div style="background-color: #f8f9fa; padding: 10px; text-align: center; border-top: 1px solid #eeeeee;">
            <button onclick="alert('Data untuk {aset['id']} ditarik ke PostgreSQL Server via FME!')"
                    style="background-color: #FFD700; color: #003366; border: 1px solid #cca100; padding: 6px 16px; font-weight: bold; border-radius: 4px; cursor: pointer; font-size: 11px; width: 100%;">
                Verifikasi Validasi Data
            </button>
        </div>

    </div>
    """

    popup_kustom = folium.Popup(html_popup, max_width=320)

    folium.Marker(
        location=aset['koordinat'],
        popup=popup_kustom,
        tooltip=f"Laporan {aset['id']}",
        icon=folium.Icon(color=aset['marker_color'], icon=aset['icon'])
    ).add_to(peta_interaktif)


# 4. MEMASUKKAN ELEMENT UI DASHBOARD MELAYANG (HEADER & LEGENDA)
header_unib_html = """
<div style="position: fixed; top: 15px; left: 70px; width: 80%; max-width: 750px; height: auto;
            background: linear-gradient(135deg, #003366, #001f3f); color: white; border: 2px solid #FFD700;
            border-radius: 10px; padding: 12px 20px; z-index: 9999; box-shadow: 0 4px 15px rgba(0,0,0,0.5);
            font-family: 'Arial', sans-serif; display: flex; align-items: center; gap: 15px;">
    <div style="font-weight: bold; font-size: 20px; color: #FFD700; border-right: 2px solid #FFD700; padding-right: 15px;">
        UNIB
    </div>
    <div>
        <h3 style="margin: 0; font-size: 15px; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;">
            Sistem Informasi Geografis (GIS) Pelaporan Aset Smart Kampus
        </h3>
        <p style="margin: 2px 0 0 0; font-size: 11px; color: #e0e0e0; font-style: italic;">
            Implementasi Pengembang Web GIS (Workflow Paperless) - Program Studi Informatika Universitas Bengkulu
        </p>
    </div>
</div>
"""

legenda_html = """
<div style="position: fixed; bottom: 30px; right: 20px; width: 220px; height: auto;
            background-color: rgba(255, 255, 255, 0.9); color: #333; border: 2px solid #003366;
            border-radius: 8px; padding: 12px; z-index: 9999; box-shadow: 0 4px 12px rgba(0,0,0,0.3);
            font-family: 'Arial', sans-serif;">
    <h4 style="margin: 0 0 8px 0; font-size: 12px; font-weight: bold; color: #003366; border-bottom: 2px solid #FFD700; padding-bottom: 4px; text-transform: uppercase;">
        Legenda Status Aset
    </h4>
    <div style="display: flex; flex-direction: column; gap: 6px; font-size: 11px;">
        <div style="display: flex; align-items: center; gap: 8px;">
            <span style="display: inline-block; width: 12px; height: 12px; background-color: red; border-radius: 50%;"></span>
            <span><b>Merah:</b> Bahaya / Rusak Berat</span>
        </div>
        <div style="display: flex; align-items: center; gap: 8px;">
            <span style="display: inline-block; width: 12px; height: 12px; background-color: orange; border-radius: 50%;"></span>
            <span><b>Oranye:</b> Perlu Penanganan</span>
        </div>
        <div style="display: flex; align-items: center; gap: 8px;">
            <span style="display: inline-block; width: 12px; height: 12px; background-color: green; border-radius: 50%;"></span>
            <span><b>Hijau:</b> Selesai / Teratasi</span>
        </div>
    </div>
    <div style="margin-top: 10px; padding-top: 8px; border-top: 1px solid #ddd; font-size: 9px; color: #666; text-align: center;">
        Informatika UNIB - Portfolio Project
    </div>
</div>
"""

# Menyematkan (inject) HTML floating ke dalam peta
peta_interaktif.get_root().html.add_child(folium.Element(header_unib_html))
peta_interaktif.get_root().html.add_child(folium.Element(legenda_html))


# 5. FITUR TAMBAHAN (Navigasi Kualitas Industri)
# Klik untuk tambah titik baru (interaksi surveyor di lapangan)
folium.ClickForMarker(
    popup="<b>SISTEM DIGITALISASI SPASIAL (OPEN SOURCE)</b><br>Titik baru siap dikirim ke basis data (PostgreSQL) tanpa kabel USB!"
).add_to(peta_interaktif)

# Inset Minimap di sudut kiri bawah
plugins.MiniMap(
    tile_layer='CartoDB positron', # Menimpa default OpenStreetMap yang diblokir
    toggle_display=True,
    position='bottomleft'
).add_to(peta_interaktif)

# Tombol untuk melihat Peta Layar Penuh
plugins.Fullscreen(position='topleft', title='Layar Penuh', title_cancel='Keluar Layar Penuh').add_to(peta_interaktif)


# 6. Tampilkan peta
peta_interaktif